# circuit

> Circuit implementation of the setpoint policy (Hebbian compass and goal weights).

In [ ]:
#| default_exp circuit

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

In [ ]:
#| export
def grid(N):
    "Grid headings in radians for indices 0..N-1."
    return 2 * np.pi * np.arange(N) / N

def rEB(c, N, kappa):
    "EB bump profile at compass index c, scaled to [0, 1] (Eq. 26)."
    e = np.exp(kappa * np.cos(grid(N) - grid(N)[c % N]))
    return (e - e.min()) / (e.max() - e.min())

def rFB(c, N):
    "FB profile at compass index c (Eq. 28)."
    return 0.5 * (np.cos(grid(N) - grid(N)[c % N]) + 1)

def fbNorm(N):
    "Normalization D in Eq. 30."
    return float((rFB(0, N) ** 2).sum())

In [ ]:
#| export
def goalStats(wG):
    "Circular mean angle, resultant length, weight sum, and strength from goal weights wG."
    N = len(wG)
    z = (wG * np.exp(1j * grid(N))).sum()
    return dict(thetaG=float(np.angle(z) % (2 * np.pi)), rG=float(abs(z)), total=float(wG.sum()),
                strength=float(abs(z) / (N / np.pi)))

def sinGoal(g, theta0, N):
    "Sinusoidal goal weights; g in [0, 0.5]."
    if g < 0 or g > 0.5:
        raise ValueError('g must be in [0, 0.5]')
    return g * np.cos(grid(N) - float(theta0)) + 0.5

def outputs(wG, c, p):
    "Policy outputs dS, nuF, rF from population sums (Eqs. 30-34)."
    N = len(wG)
    q = N // 4
    D = fbNorm(N)
    def rO(shift):
        return float((rFB(c + shift, N) * wG).sum()) / D
    rL, rR = rO(+q), rO(-q)
    rF = rO(N // 2) + p.nuMax - wG.sum() / D
    dS = 0.5 * (1 + rR - rL)
    return dict(dS=dS, nuF=max(rF, 0.0) + p.BF, rF=rF)

In [ ]:
#| export
def ringActivity(a, N, symmetric):
    "Ring-neuron activity at arena heading a (one or two bumps)."
    act = np.zeros(N)
    bumps = [int(a) % N]
    if symmetric:
        bumps.append((int(a) + N // 2) % N)
    for b in bumps:
        act[b] = 1.0
        for k in (b - 1, b + 1):
            act[k % N] = max(act[k % N], 0.5)
    return act

def jumpProb(wC, act, c):
    "Jump probability pJ from net compass inhibition (Eq. 27)."
    net = act @ wC
    N = len(net)
    x, y = float(net[c % N]), float(net[(c + N // 2) % N])
    return 0.5 if x + y == 0 else x / (x + y)

def hebb(target, w):
    "Hebbian delta toward target with [0, 1] limits (Eq. 39, R > 0)."
    return np.maximum(target - w, 0) * (w < 1) - np.maximum(w - target, 0) * (w > 0)

def antiHebbGoal(target, w):
    "Anti-Hebbian delta away from target (Eq. 39, R < 0)."
    return -np.maximum(target - w, 0) * (w > 0) + np.maximum(w - target, 0) * (w < 1)

def compassStep(wC, act, c, size, p):
    "One saccade grid step of anti-Hebbian compass learning."
    target = 1 - rEB(c, len(wC), p.kappaEB)
    dw = p.alphaC * size ** 2 * act[:, None] * hebb(target[None, :], wC)
    return np.clip(wC + dw, 0, 1)

def goalStep(wG, c, R, p):
    "One dtG step of goal learning at compass heading c."
    if R == 0:
        return wG
    t = rFB(c, len(wG))
    fn = hebb if R > 0 else antiHebbGoal
    return np.clip(wG + p.alphaG * fn(t, wG), 0, 1)

In [ ]:
from math import pi
import math
from fastcore.test import test_eq, test_close, test_fail
from explRL.params import Params

N = 32
p = Params()
th = grid(N)
reb = rEB(5, N, p.kappaEB)
test_close(reb.min(), 0.0)
test_close(reb.max(), 1.0)
test_close(reb[5], 1.0)
test_close(fbNorm(32), 12.0)

wG = sinGoal(0.5, th[8], N)
gs = goalStats(wG)
test_close(gs['thetaG'], th[8], eps=1e-9)
test_close(gs['rG'], 0.5 * N / 2, eps=1e-9)
test_fail(lambda: sinGoal(0.6, 0.0, N), contains='g must')
test_fail(lambda: sinGoal(-0.1, 0.0, N), contains='g must')

# AC-2 closed form
D = fbNorm(N)
rG = gs['rG']
thetaG = gs['thetaG']
for c in range(N):
    out = outputs(wG, c, p)
    tc = th[c]
    dS_cf = 0.5 + rG / (2 * D) * np.sin(tc - thetaG)
    test_close(out['dS'], dS_cf, eps=1e-9)
ds_all = np.array([outputs(wG, c, p)['dS'] for c in range(N)])
test_eq(int(ds_all.argmax()), 16)
test_close(ds_all.min(), 1/6, eps=1e-9)
test_close(ds_all.max(), 5/6, eps=1e-9)
nu_all = np.array([outputs(wG, c, p)['nuF'] for c in range(N)])
test_eq(int(nu_all.argmin()), 8)

# section 10 table spot checks
for g, rG_exp in [(0.25, 4.0), (0.5, 8.0)]:
    w = sinGoal(g, 0.0, N)
    test_close(goalStats(w)['rG'], rG_exp, eps=1e-6)
w1 = np.ones(N)
test_close(goalStats(w1)['strength'], 0.0, eps=1e-9)
test_close(outputs(w1, 0, p)['rF'], -0.233, eps=1e-3)

# ring activity
act_a = ringActivity(0, N, False)
test_eq(int((act_a > 0).sum()), 3)
test_close(act_a[[0, 1, N-1]].tolist(), [1.0, 0.5, 0.5])
act_w = ringActivity(N - 1, N, False)
test_close(act_w[[N - 1, 0, N - 2]].tolist(), [1.0, 0.5, 0.5])
act_s = ringActivity(0, N, True)
test_eq(int((act_s > 0).sum()), 6)
wC0 = np.ones((N, N)) * 0.5
test_close(jumpProb(wC0, act_a, 0), 0.5)
test_close(jumpProb(np.zeros((N, N)), act_a, 0), 0.5)

# AC-4 goal plasticity
wG0 = sinGoal(0.3, th[0], N).copy()
c_fix = 8

def _ang_to_c(gs, c):
    return abs((gs['thetaG'] - th[c] + pi) % (2 * pi) - pi)

w_r = goalStep(wG0, c_fix, 1, p)
w_p = goalStep(wG0, c_fix, -1, p)
w_z = goalStep(wG0, c_fix, 0, p)
test_close(w_z, wG0)
assert _ang_to_c(goalStats(w_r), c_fix) < _ang_to_c(goalStats(wG0), c_fix)
assert _ang_to_c(goalStats(w_p), c_fix) > _ang_to_c(goalStats(wG0), c_fix)
assert goalStats(goalStep(wG0, 0, 1, p))['strength'] > goalStats(wG0)['strength']
assert w_r.min() >= 0 and w_r.max() <= 1

# AC-5 compass plasticity
from explRL.envs import ZoneArena
rng = np.random.default_rng(0)
wC = rng.uniform(0, 1, (N, N))
env = ZoneArena(N=N, symmetric=False)
mirror = 10
def mean_pj(wC):
    return float(np.mean([jumpProb(wC, env.ringActivity(a), (mirror - a) % N) for a in range(N)]))
p0 = mean_pj(wC)
for _ in range(1000):
    direction = 1 if rng.random() < 0.5 else -1
    size = float(np.deg2rad(rng.lognormal(p.muS, p.sigmaS)))
    steps = max(1, int(math.ceil(size / (2 * pi / N))))
    a = int(rng.integers(N))
    c = (mirror - a) % N
    for _ in range(steps):
        wC = compassStep(wC, env.ringActivity(a), c, size, p)
        a = (a + direction) % N
        c = (c - direction) % N
assert p0 > 0.45
assert mean_pj(wC) < 0.15
assert wC.min() >= 0 and wC.max() <= 1

for w0 in (np.zeros((N, N)), np.ones((N, N))):
    w1 = compassStep(w0, ringActivity(3, N, False), 3, 0.1, p)
    assert w1.min() >= 0 and w1.max() <= 1
    w2 = goalStep(np.zeros(N), 4, 1, p)
    assert w2.min() >= 0 and w2.max() <= 1


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()